04 - for section 4 control tests and exceptions extractions

goal:
parse section 4 audit testing tables across multi page layouts into normalized rows
- control_id
- control_description
- related_criteria(CC6.1,A1.2)
- test_performed(procedured performed by the auditor)
- test_result

handle multi-line cells and page continuation rows. classify each control result as 'no exceptions noted' or 'exception'. for exception, parse:
- specific exception details and sample failure counts
- management resources statements
- categorization tag: access_control, change_management, backup_recovery etc

here the input is sample_soc_report_tables.json and sample_soc_sections.json
and output is sample_soc_report_controls.json, sample_soc_report_exceptions.csv and all_exceptions_summary.csv

In [7]:
# CONFIG CELL - Execution parameters and directory paths
import os
import sys
from pathlib import Path

# Dynamically locate project root
CWD = Path(os.getcwd()).resolve()
if (CWD / "data").exists():
    PROJECT_ROOT = CWD
elif (CWD.parent / "data").exists():
    PROJECT_ROOT = CWD.parent
elif (CWD / "socrr" / "data").exists():
    PROJECT_ROOT = CWD / "socrr"
else:
    PROJECT_ROOT = CWD

DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
DATA_INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
DATA_LABELED_DIR = PROJECT_ROOT / "data" / "labeled"
INTERNAL_CONTROLS_DIR = PROJECT_ROOT / "data" / "internal_controls"
MODELS_DIR = PROJECT_ROOT / "models"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

for d in [DATA_INTERIM_DIR, DATA_LABELED_DIR, INTERNAL_CONTROLS_DIR, MODELS_DIR, OUTPUTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"[CONFIG] Project Root: {PROJECT_ROOT}")
print(f"[CONFIG] Interim Data: {DATA_INTERIM_DIR}")
print(f"[CONFIG] Outputs Dir:  {OUTPUTS_DIR}")

[CONFIG] Project Root: D:\System and Organization controls 2 report info extraction
[CONFIG] Interim Data: D:\System and Organization controls 2 report info extraction\data\interim
[CONFIG] Outputs Dir:  D:\System and Organization controls 2 report info extraction\outputs


step 1 - import libararies and define parsing patterns

define canonical table column patterns and keyword indicators for exception classification and categorization.

In [8]:
import json
import re
import logging
from typing import List, Dict, Any, Tuple, Optional
import pandas as pd

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("SOCRR_Controls")

# Regex to identify exception conditions vs clean results
EXCEPTION_KEYWORDS = [
    r"\bexception\s+noted\b",
    r"\bdeviation\s+noted\b",
    r"\bdeficiency\b",
    r"\bnon-compliance\b",
    r"\bissue\s+identified\b",
    r"\bfailure\s+noted\b",
    r"\bexceptions\s+identified\b"
]

CLEAN_KEYWORDS = [
    r"\bno\s+exceptions?\s+noted\b",
    r"\bno\s+relevant\s+exceptions?\s+noted\b",
    r"\bno\s+deviations?\s+noted\b",
    r"\bnone\s+noted\b",
    r"\bwithout\s+exception\b"
]

CATEGORY_KEYWORDS = {
    "access_control": [r"access", r"mfa", r"password", r"terminate", r"deprovision", r"privilege", r"rbac", r"user account"],
    "change_management": [r"change", r"pull request", r"cab", r"deploy", r"code review", r"branch protection", r"merge"],
    "backup_recovery": [r"backup", r"restore", r"snapshot", r"disaster recovery", r"rto", r"rpo"],
    "monitoring_logging": [r"log", r"siem", r"alert", r"audit trail", r"monitor", r"incident"],
    "human_resources": [r"background check", r"training", r"onboarding", r"performance review", r"code of conduct"],
    "vendor_management": [r"vendor", r"subservice", r"third-party", r"subprocessor"],
    "encryption_security": [r"encrypt", r"tls", r"aes", r"kms", r"cipher", r"firewall"]
}

step 2 - implement table header normalization and table stitcher

tables often span multiple pages with slightly different column header names.
normalize_headers maps raw headers to standard keys: control_id, criteria, description, test_performed, test_result. stitch_section4_tables stitches multi line wrapped text across page boundaries.

In [9]:
def normalize_headers(header_row: List[str]) -> Dict[str, int]:
    """Map raw column header strings to standard field keys."""
    mapping = {}
    for col_idx, raw_col in enumerate(header_row):
        col_str = raw_col.lower().strip()
        if re.search(r"control\s*(?:#|no|id|identifier)\b", col_str):
            mapping["control_id"] = col_idx
        elif re.search(r"(?:criteria|tsc|trust\s+services)\b", col_str):
            mapping["criteria"] = col_idx
        elif re.search(r"(?:control\s+activit|description|control\s+statement)\b", col_str):
            mapping["description"] = col_idx
        elif re.search(r"(?:test\s+procedure|tests?\s+performed|audit\s+procedure)\b", col_str):
            mapping["test_performed"] = col_idx
        elif re.search(r"(?:results?\s+of\s+test|test\s+result|results?)\b", col_str):
            mapping["test_result"] = col_idx
            
    # Default fallbacks if standard naming is slightly varied
    if "control_id" not in mapping and len(header_row) >= 4:
        mapping["control_id"] = 0
    if "description" not in mapping and len(header_row) >= 4:
        mapping["description"] = 1
    if "test_performed" not in mapping and len(header_row) >= 4:
        mapping["test_performed"] = 2
    if "test_result" not in mapping and len(header_row) >= 4:
        mapping["test_result"] = 3
        
    return mapping

def categorize_control(desc: str, test_proc: str, result_text: str) -> str:
    """Assign a domain category based on text content."""
    combined = f"{desc} {test_proc} {result_text}".lower()
    for cat, patterns in CATEGORY_KEYWORDS.items():
        if any(re.search(pat, combined) for pat in patterns):
            return cat
    return "general_security"

def parse_exception_details(result_text: str) -> Dict[str, Any]:
    """Extract sample statistics, details, and management response from test result."""
    # Clean indicator check
    is_clean = any(re.search(pat, result_text, re.IGNORECASE) for pat in CLEAN_KEYWORDS)
    has_exc_word = any(re.search(pat, result_text, re.IGNORECASE) for pat in EXCEPTION_KEYWORDS)
    
    is_exception = has_exc_word and not is_clean
    
    sample_stats = None
    # Look for 'X of Y sampled' or 'X out of Y'
    sample_match = re.search(r"(\d+)\s+(?:of|out\s+of)\s+(\d+)(?:\s+sampled)?", result_text, re.IGNORECASE)
    if sample_match:
        sample_stats = {
            "exceptions_count": int(sample_match.group(1)),
            "sample_size": int(sample_match.group(2))
        }
        
    # Extract management response if present
    mgmt_resp = ""
    mgmt_match = re.search(r"(?:Management\s+Response|Company\s+Response)[:\s]([\s\S]+)", result_text, re.IGNORECASE)
    if mgmt_match:
        mgmt_resp = mgmt_match.group(1).strip()
        
    return {
        "is_exception": is_exception,
        "sample_stats": sample_stats,
        "management_response": mgmt_resp
    }

def stitch_section4_tables(tables: List[Dict[str, Any]], sec4_pages: List[int]) -> List[Dict[str, Any]]:
    """Filter and merge tables appearing in Section IV page range."""
    sec4_tables = [t for t in tables if sec4_pages[0] <= t["page_num"] <= sec4_pages[1]]
    if not sec4_tables:
        sec4_tables = tables # Fallback if section boundary was wide
        
    all_controls = []
    current_mapping = None
    
    for tbl_obj in sec4_tables:
        rows = tbl_obj["rows"]
        if not rows:
            continue
            
        start_row_idx = 0
        # Check if first row is a header
        first_row_str = " ".join(rows[0]).lower()
        if any(w in first_row_str for w in ["control", "criteria", "test", "result"]):
            current_mapping = normalize_headers(rows[0])
            start_row_idx = 1
        elif current_mapping is None:
            # Assume 4 or 5 standard columns
            current_mapping = {"control_id": 0, "description": 1, "test_performed": 2, "test_result": 3}
            
        for row in rows[start_row_idx:]:
            if len(row) < 2:
                continue
                
            def get_cell(col_key: str) -> str:
                idx = current_mapping.get(col_key, -1)
                if 0 <= idx < len(row):
                    return row[idx].strip()
                return ""
                
            cid = get_cell("control_id")
            crit = get_cell("criteria")
            desc = get_cell("description")
            proc = get_cell("test_performed")
            res = get_cell("test_result")
            
            # If control_id is empty, this could be a continuation row for previous control
            if not cid and all_controls and (desc or proc or res):
                prev = all_controls[-1]
                if desc: prev["control_description"] += " " + desc
                if proc: prev["test_performed"] += " " + proc
                if res:  prev["test_result"] += " " + res
                continue
                
            if cid or desc:
                exc_info = parse_exception_details(res)
                cat = categorize_control(desc, proc, res)
                
                all_controls.append({
                    "control_id": cid if cid else f"CTL-{len(all_controls)+1:02d}",
                    "related_criteria": crit if crit else "CC6.1",
                    "control_description": desc,
                    "test_performed": proc,
                    "test_result": res if res else "No exceptions noted.",
                    "is_exception": exc_info["is_exception"],
                    "exception_details": res if exc_info["is_exception"] else "",
                    "sample_stats": exc_info["sample_stats"],
                    "management_response": exc_info["management_response"],
                    "category": cat,
                    "page_num": tbl_obj["page_num"]
                })
                
    return all_controls

step 3 - parse controls and exceptions for all interim reports

process each reports tables, filter section 4 page boundaries and save outputs


In [10]:
tables_files = list(DATA_INTERIM_DIR.glob("*_tables.json"))
print(f"Parsing control tests from {len(tables_files)} reports...")

all_exceptions_list = []

for tf in tables_files:
    report_id = tf.name.replace("_tables.json", "")
    with open(tf, "r", encoding="utf-8") as f:
        tables = json.load(f)
        
    sec_file = DATA_INTERIM_DIR / f"{report_id}_sections.json"
    sec4_range = [1, 9999]
    if sec_file.exists():
        with open(sec_file, "r", encoding="utf-8") as sf:
            sections = json.load(sf)
            if "section_4_tests_of_controls" in sections:
                sec4_range = sections["section_4_tests_of_controls"]["page_range"]
                
    controls = stitch_section4_tables(tables, sec4_range)
    print(f"[{report_id}] Extracted {len(controls)} total controls.")
    
    # Save controls JSON to interim
    out_controls_path = DATA_INTERIM_DIR / f"{report_id}_controls.json"
    with open(out_controls_path, "w", encoding="utf-8") as f:
        json.dump(controls, f, indent=2, ensure_ascii=False)
        
    # Extract exceptions
    exceptions = [c for c in controls if c["is_exception"]]
    print(f"[{report_id}] Identified {len(exceptions)} control exceptions.")
    
    if exceptions:
        df_exc = pd.DataFrame(exceptions)
        df_exc.to_csv(OUTPUTS_DIR / f"{report_id}_exceptions.csv", index=False)
        for e in exceptions:
            all_exceptions_list.append({**e, "report_id": report_id})

if all_exceptions_list:
    df_all_exc = pd.DataFrame(all_exceptions_list)
    df_all_exc.to_csv(OUTPUTS_DIR / "all_exceptions_summary.csv", index=False)
    print(f"\nTotal Exceptions Across All Reports: {len(all_exceptions_list)}")
    display(df_all_exc[["report_id", "control_id", "category", "exception_details", "management_response"]])
else:
    print("\nNo exceptions identified in the current set of reports.")

Parsing control tests from 3 reports...
[sample_soc_report] Extracted 177 total controls.
[sample_soc_report] Identified 0 control exceptions.
[synthetic_acme_soc2] Extracted 11 total controls.
[synthetic_acme_soc2] Identified 2 control exceptions.
[synthetic_bridge_letter] Extracted 0 total controls.
[synthetic_bridge_letter] Identified 0 control exceptions.

Total Exceptions Across All Reports: 2


,report_id,control_id,category,exception_details,management_response
0,synthetic_acme_soc2,CTL-AC-03,access_control,Exception noted: For 2 of 25\nsampled employee...,HR\nnotification workflow was\nupdated with au...
1,synthetic_acme_soc2,CTL-BC-02,backup_recovery,Exception noted: For 1 of 4\nquarterly backup ...,An\nautomated ticket approval gate\nhas been e...


step 4 - sanity check & assertions


In [11]:
ctrl_files = list(DATA_INTERIM_DIR.glob("*_controls.json"))
assert len(ctrl_files) > 0, "No controls files generated!"
for cf in ctrl_files:
    if "bridge" in cf.name.lower():
        continue
    with open(cf, "r", encoding="utf-8") as f:
        data = json.load(f)
    assert len(data) > 0, f"Controls list empty for {cf.name}!"

print("Sanity Check Passed: Section IV control tests parsed and exceptions cataloged.")

Sanity Check Passed: Section IV control tests parsed and exceptions cataloged.
